# This document Combines and compared Controller and Robot Classes 

This copy adds the 1 ms friction estimator (`ControllerModels_Estimator.ipynb`): controller type `'mpc_est'`. Everything else is Dana's.


## Comparison Simulation Types

1. Single predefined motion path
    * Slow and Fast path duration comparisons 
    * Reducing time duration analysis


In [ ]:
import time

class CodeTimer:
    """
    A versatile timer class for measuring elapsed execution time with 
    support for starting, pausing, resuming, and resetting.
    """
    def __init__(self):
        self.reset()

    def start(self):
        """Starts the timer from a fresh state or restarts it."""
        self._start_time = time.perf_counter()
        self._elapsed_time = 0.0
        self._is_running = True

    def pause(self):
        """Pauses the timer if it is currently running."""
        if self._is_running:
            self._elapsed_time += time.perf_counter() - self._start_time
            self._is_running = False

    def resume(self):
        """Resumes the timer if it was paused."""
        if not self._is_running:
            self._start_time = time.perf_counter()
            self._is_running = True

    def reset(self):
        """Resets all timer counters and stops the timer."""
        self._start_time = 0.0
        self._elapsed_time = 0.0
        self._is_running = False

    @property
    def elapsed(self) -> float:
        """Returns the total elapsed time in seconds, updating live if running."""
        if self._is_running:
            return self._elapsed_time + (time.perf_counter() - self._start_time)
        return self._elapsed_time


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# =======================================================================
# 0. BRING IN FUNCTIONS FROM OTHER NOTEBOOKS
# =======================================================================

from IPython.utils.capture import capture_output
global simulation_parameters, PathGenerator, OmniRobotSimulationExplicit
# This forces the native Jupyter runner to execute the notebook silently
with capture_output():
    %run "PlantModels.ipynb"
    %run "ControllerModels_Estimator.ipynb"
    %run "PathGeneration.ipynb"


## Simulation Runner Function 

In [ ]:
# paramaters and classed are defined in xxxx.ipynb and imported above, so we need to declare them as global to avoid "undefined variable" errors
global model_choice, simulation_parameters, PathGenerator
global OmniRobotSimulationExplicit, OmniRobotStateExplicit, OmniRobotSimulationODESolve, OmniRobotStateODESolve
global CascadedPIDController, MPCControllerSimple



def execute_simulation(plant_parm:dict, controller_type: str, ref_path:dict) -> dict:
    """Executes closed-loop simulation for either PID or MPC over duration T."""
    
    if model_choice == 1:
        plant = OmniRobotSimulationExplicit(dict(plant_parm, lugre_deflection=100e-6), random_seed=42)   # explicit plant: 100 um bristle (2 um is too stiff for it)
        plant_state = OmniRobotStateExplicit()  
        observable_state = plant.get_observable_state(plant_state)  
    elif model_choice == 2:   
        plant = OmniRobotSimulationODESolve(plant_parm, random_seed=42)
        plant_state = OmniRobotStateODESolve()
        observable_state = plant.get_observable_state(plant_state)
    
    if controller_type.lower() == 'pid':
        ctrl = CascadedPIDController(
            plant_parm,
            pos_pid_gains={'kp': 1.5, 'ki': 0.1, 'kd': 0.05},
            wheel_pid_gains={'kp': 0.15, 'ki': 0.8, 'kd': 0.0},
            dt=0.02, a_max=3.5, alpha_max=7.0, 
            k_theta=3.0, voltage_limit=12.0
        )
    elif controller_type.lower() == 'mpc_est':
        # ADDED: Dana's MPC with the 1 ms friction estimator, same settings
        ctrl = MPCControllerFrictionEstimator(
            plant_parm, horizon_K=8, dt=0.02,
            q_pose=(100.0, 100.0, 60.0), q_vel=(15.0, 15.0, 8.0),
            r_voltage=0.015, r_slew=0.008, 
            voltage_limit=12.0, current_limit = 20.0, delta_v_max=5.0
            )
    else:
        ctrl = MPCControllerSimple(
            plant_parm, horizon_K=8, dt=0.02,
            q_pose=(100.0, 100.0, 60.0), q_vel=(15.0, 15.0, 8.0),
            r_voltage=0.015, r_slew=0.008, 
            voltage_limit=12.0, current_limit = 20.0, delta_v_max=5.0
            )

    # Extract reference trajectory data
    t_ref = ref_path['t']
    pos_ref = ref_path['pos']
    vel_ref = ref_path['vel']
    N = len(t_ref)
    duration = t_ref[-1] - t_ref[0]
    
    # Precompute local reference velocity vectors
    V_local_ref = np.zeros_like(vel_ref)
    for i in range(N):
        th_i = pos_ref[i, 2]
        cos_t = np.cos(th_i)
        sin_t = np.sin(th_i)
        R_t = np.array([[cos_t, -sin_t, 0.0], [sin_t, cos_t, 0.0], [0.0, 0.0, 1.0]])
        V_local_ref[i] = R_t.T @ vel_ref[i]

    # Initialize logging arrays   
    log_pos = []
    log_v = []
    log_V = []
    log_vs = []
    log_opt = []
    
    dt_ctrl = ctrl.dt
    substeps = int(round(dt_ctrl / plant_parm['dt'])) # holding substeps

    timer = CodeTimer()
    timer.start()
    for k in range(N):
        observable_state = plant.get_observable_state(plant_state)
        
        p_d = pos_ref[k]
        v_d = vel_ref[k]
        
        if controller_type.lower() == 'pid':
            u = ctrl.compute(p_d, observable_state)
            is_opt = True
        else:
            if k + ctrl.K + 1 <= N:
                p_h = pos_ref[k : k + ctrl.K + 1]
                v_h = V_local_ref[k : k + ctrl.K + 1]
            else:
                pad = (k + ctrl.K + 1) - N
                p_h = np.vstack([pos_ref[k:], np.tile(pos_ref[-1], (pad, 1))])
                v_h = np.vstack([V_local_ref[k:], np.zeros((pad, 3))])
            u, info = ctrl.compute_optimal_u0(measured_State=observable_state, path_ref_horizon=p_h, vel_ref_horizon=v_h, return_info=True)
            is_opt = info['optimal']

        #loop through the substeps to integrate the plant dynamics
        timer.pause()
        for _ in range(substeps):
            plant_state = plant.step(plant_state, u)
            if controller_type.lower() == 'mpc_est':
                # ADDED: 1 ms friction estimator. On the robot it runs on its own core, so it is not timed here
                ctrl.estimator_update(plant.get_observable_state(plant_state))

        
        # calculate data from plant_state and observable_state
        v_contact = plant.H @ plant_state.v_local
        v_s = v_contact - plant.p['r'] * plant_state.omega

        # log data for analysis  
        log_pos.append(plant_state.pos_global.copy())
        log_v.append(plant_state.v_global.copy())
        log_V.append(u.copy())
        log_vs.append(v_s.copy())
        log_opt.append(is_opt)
        timer.resume()

    timer.pause()    
    elapsed = timer.elapsed
    log_pos = np.array(log_pos)
    log_v = np.array(log_v)
    log_V = np.array(log_V)
    log_vs = np.array(log_vs)
    log_opt = np.array(log_opt)
    
    dist_err = np.hypot(pos_ref[:, 0] - log_pos[:, 0], pos_ref[:, 1] - log_pos[:, 1])
    th_err = np.abs((pos_ref[:, 2] - log_pos[:, 2] + np.pi) % (2.0 * np.pi) - np.pi)
    
    return {
        'type': controller_type.upper(), 'duration': duration,
        't': t_ref, 'pos_ref': pos_ref, 'vel_ref': vel_ref, 'V_local_ref': V_local_ref,
        'pos': log_pos, 'v': log_v, 'V': log_V, 'vs': log_vs,
        'dist_err': dist_err, 'th_err': th_err, 'log_opt': log_opt,
        'elapsed': elapsed
    }




In [ ]:
# paramaters and classed are defined in xxxx.ipynb and imported above, so we need to declare them as global to avoid "undefined variable" errors
global model_choice, simulation_parameters, PathGenerator
global OmniRobotSimulationExplicit, OmniRobotStateExplicit, OmniRobotSimulationODESolve, OmniRobotStateODESolve
global CascadedPIDController, MPCControllerSimple


# =======================================================================
# 1. SETUP SIMULATION PLANT & CONTROLLERS
# =======================================================================

path_time_fast = 8.0
path_time_slow = 4.0
sim_params = simulation_parameters.copy()

pg_comp = PathGenerator(dt=0.02)
start_pose = np.array([0.0, 0.0, 0.0])
target_pose = np.array([3.0, 3.0, np.pi / 4.0])
ref_path_slow = pg_comp.generate_coupled_segment(start_pose, target_pose, duration=path_time_fast)
ref_path_fast = pg_comp.generate_coupled_segment(start_pose, target_pose, duration=path_time_slow)


print("=======================================================================")
print("  🚀 EXECUTING COMPARATIVE BENCHMARK (CASCADED PID vs. CONVEX MPC)     ")
print("=======================================================================")
print(f" [1/4] Running Cascaded PID under Nominal Motion (T = {path_time_slow}s)...")
res_pid_nom = execute_simulation(sim_params,'pid',ref_path_slow)

print(f" [2/4] Running Convex MPC under Nominal Motion (T = {path_time_slow}s)...")
res_mpc_nom = execute_simulation(sim_params,'mpc',ref_path_slow)

print(f" [3/4] Running Cascaded PID under Fast Motion (T = {path_time_fast} s)...")
res_pid_fast = execute_simulation(sim_params,'pid',ref_path_fast)

print(f" [4/4] Running Convex MPC under Fast Motion (T = {path_time_fast} s)...")
res_mpc_fast = execute_simulation(sim_params,'mpc',ref_path_fast)

print("All simulations completed successfully!")

# =======================================================================
# 2. PRINT COMPREHENSIVE PERFORMANCE METRICS TABLE
# =======================================================================
print("\n" + "="*88)
print("             🏆 CASCADED PID vs. CONVEX MPC PERFORMANCE SUMMARY        ")
print("="*88)
print(f"{'Metric':<32} | {'PID (T=8.0s)':<12} | {'MPC (T=8.0s)':<12} | {'PID (T=4.0s)':<12} | {'MPC (T=4.0s)':<12}")
print("-" * 88)
print(f"{'Final Position Error':<32} | {res_pid_nom['dist_err'][-1]*100:6.2f} cm     | {res_mpc_nom['dist_err'][-1]*100:6.2f} cm     | {res_pid_fast['dist_err'][-1]*100:6.2f} cm     | {res_mpc_fast['dist_err'][-1]*100:6.2f} cm")
print(f"{'Final Heading Error':<32} | {np.degrees(res_pid_nom['th_err'][-1]):6.2f}°      | {np.degrees(res_mpc_nom['th_err'][-1]):6.2f}°      | {np.degrees(res_pid_fast['th_err'][-1]):6.2f}°      | {np.degrees(res_mpc_fast['th_err'][-1]):6.2f}°")
print(f"{'Maximum Position Error':<32} | {np.max(res_pid_nom['dist_err'])*100:6.2f} cm     | {np.max(res_mpc_nom['dist_err'])*100:6.2f} cm     | {np.max(res_pid_fast['dist_err'])*100:6.2f} cm     | {np.max(res_mpc_fast['dist_err'])*100:6.2f} cm")
print(f"{'RMS Position Error':<32} | {np.sqrt(np.mean(res_pid_nom['dist_err']**2))*100:6.2f} cm     | {np.sqrt(np.mean(res_mpc_nom['dist_err']**2))*100:6.2f} cm     | {np.sqrt(np.mean(res_pid_fast['dist_err']**2))*100:6.2f} cm     | {np.sqrt(np.mean(res_mpc_fast['dist_err']**2))*100:6.2f} cm")
print(f"{'Peak Commanded Voltage':<32} | {np.max(np.abs(res_pid_nom['V'])):6.1f} V      | {np.max(np.abs(res_mpc_nom['V'])):6.1f} V      | {np.max(np.abs(res_pid_fast['V'])):6.1f} V      | {np.max(np.abs(res_mpc_fast['V'])):6.1f} V")
print(f"{'Peak Contact Slip Velocity':<32} | {np.max(np.abs(res_pid_nom['vs']))*1000:6.1f} mm/s   | {np.max(np.abs(res_mpc_nom['vs']))*1000:6.1f} mm/s   | {np.max(np.abs(res_pid_fast['vs']))*1000:6.1f} mm/s   | {np.max(np.abs(res_mpc_fast['vs']))*1000:6.1f} mm/s")
print(f"{'Optimal Solution Rate':<32} | {'100.0%':<12} | {'100.0%':<12} | {'100.0%':<12} | {'100.0%':<12}")
print(f"{'Execution Time per Step':<32} | {res_pid_nom['elapsed']/len(res_pid_nom['t'])*1000:6.2f} ms    | {res_mpc_nom['elapsed']/len(res_mpc_nom['t'])*1000:6.2f} ms    | {res_pid_fast['elapsed']/len(res_pid_fast['t'])*1000:6.2f} ms    | {res_mpc_fast['elapsed']/len(res_mpc_fast['t'])*1000:6.2f} ms")
print("="*88)

# =======================================================================
# 3. FIGURE 1: SIDE-BY-SIDE 2D TRAJECTORY TRACKING
# =======================================================================
fig, axs = plt.subplots(2, 2, figsize=(14, 13))

def plot_trajectory_panel(ax, res, title, path_color):
    arena = plt.Rectangle((-4.5, -4.5), 9.0, 9.0, fill=False, edgecolor='black', linestyle='--', lw=1.5, label='9m × 9m Arena')
    ax.add_patch(arena)
    
    pos_ref = res['pos_ref']
    pos = res['pos']
    
    ax.plot(pos_ref[:, 0], pos_ref[:, 1], 'k:', lw=1.8, alpha=0.7, label='Reference Setpoint')
    ax.plot(pos[:, 0], pos[:, 1], color=path_color, lw=2.5, label='Actual Robot Path')
    
    ax.plot(pos_ref[0, 0], pos_ref[0, 1], 'gs', markersize=9, label='Start: (0, 0, 0)')
    ax.plot(pos_ref[-1, 0], pos_ref[-1, 1], 'r*', markersize=13, label=r'Target: (3, 3, $\pi/4$)')
    
    # Overlay heading tangent directors (+y_r forward)
    step_jump = max(1, len(pos) // 12)
    indices = np.arange(0, len(pos), step_jump)
    for idx in indices:
        xi, yi, thi = pos[idx, 0], pos[idx, 1], pos[idx, 2]
        u_fwd = -np.sin(thi) * 0.32
        v_fwd =  np.cos(thi) * 0.32
        ax.annotate('', xy=(xi + u_fwd, yi + v_fwd), xytext=(xi, yi),
                    arrowprops=dict(arrowstyle="->", color="crimson", lw=1.5))
                    
    ax.set_xlim(-0.5, 3.8)
    ax.set_ylim(-0.5, 3.8)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Global X Position (m)', fontsize=10)
    ax.set_ylabel('Global Y Position (m)', fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower right', fontsize=8.5)

plot_trajectory_panel(axs[0, 0], res_pid_nom, 'Cascaded PID — Nominal Motion (T = 8.0 s)', 'royalblue')
plot_trajectory_panel(axs[0, 1], res_mpc_nom, 'Convex MPC — Nominal Motion (T = 8.0 s)', 'darkorange')
plot_trajectory_panel(axs[1, 0], res_pid_fast, 'Cascaded PID — Fast Motion (T = 4.0 s)', 'royalblue')
plot_trajectory_panel(axs[1, 1], res_mpc_fast, 'Convex MPC — Fast Motion (T = 4.0 s)', 'darkorange')

plt.tight_layout()
plt.show()

# =======================================================================
# 4. FIGURE 2: POSITION & HEADING TRACKING ERRORS
# =======================================================================
fig, axs = plt.subplots(2, 2, figsize=(14, 9))

# Top-Left: Nominal Position Error
axs[0, 0].plot(res_pid_nom['t'], res_pid_nom['dist_err'] * 100.0, 'royalblue', lw=2.0, label=f'Cascaded PID (Peak: {np.max(res_pid_nom["dist_err"])*100:.1f} cm, End: {res_pid_nom["dist_err"][-1]*100:.2f} cm)')
axs[0, 0].plot(res_mpc_nom['t'], res_mpc_nom['dist_err'] * 100.0, 'darkorange', lw=2.2, label=f'Convex MPC (Peak: {np.max(res_mpc_nom["dist_err"])*100:.1f} cm, End: {res_mpc_nom["dist_err"][-1]*100:.2f} cm)')
axs[0, 0].set_title('Nominal Motion (T = 8.0 s) — Euclidean Position Error', fontsize=11, fontweight='bold')
axs[0, 0].set_ylabel('Position Error (cm)', fontsize=10)
axs[0, 0].grid(True, linestyle=':', alpha=0.6)
axs[0, 0].legend(loc='upper right', fontsize=8.5)

# Top-Right: Fast Position Error
axs[0, 1].plot(res_pid_fast['t'], res_pid_fast['dist_err'] * 100.0, 'royalblue', lw=2.0, label=f'Cascaded PID (Peak: {np.max(res_pid_fast["dist_err"])*100:.1f} cm, End: {res_pid_fast["dist_err"][-1]*100:.2f} cm)')
axs[0, 1].plot(res_mpc_fast['t'], res_mpc_fast['dist_err'] * 100.0, 'darkorange', lw=2.2, label=f'Convex MPC (Peak: {np.max(res_mpc_fast["dist_err"])*100:.1f} cm, End: {res_mpc_fast["dist_err"][-1]*100:.2f} cm)')
axs[0, 1].set_title('Fast Motion (T = 4.0 s) — Euclidean Position Error', fontsize=11, fontweight='bold')
axs[0, 1].set_ylabel('Position Error (cm)', fontsize=10)
axs[0, 1].grid(True, linestyle=':', alpha=0.6)
axs[0, 1].legend(loc='upper right', fontsize=8.5)

# Bottom-Left: Nominal Heading Error
axs[1, 0].plot(res_pid_nom['t'], np.degrees(res_pid_nom['th_err']), 'royalblue', lw=2.0, label=f'Cascaded PID (End: {np.degrees(res_pid_nom["th_err"][-1]):.2f}°)')
axs[1, 0].plot(res_mpc_nom['t'], np.degrees(res_mpc_nom['th_err']), 'darkorange', lw=2.2, label=f'Convex MPC (End: {np.degrees(res_mpc_nom["th_err"][-1]):.2f}°)')
axs[1, 0].set_title('Nominal Motion (T = 8.0 s) — Heading Angle Error', fontsize=11, fontweight='bold')
axs[1, 0].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 0].set_ylabel('Heading Error (°)', fontsize=10)
axs[1, 0].grid(True, linestyle=':', alpha=0.6)
axs[1, 0].legend(loc='upper right', fontsize=8.5)

# Bottom-Right: Fast Heading Error
axs[1, 1].plot(res_pid_fast['t'], np.degrees(res_pid_fast['th_err']), 'royalblue', lw=2.0, label=f'Cascaded PID (End: {np.degrees(res_pid_fast["th_err"][-1]):.2f}°)')
axs[1, 1].plot(res_mpc_fast['t'], np.degrees(res_mpc_fast['th_err']), 'darkorange', lw=2.2, label=f'Convex MPC (End: {np.degrees(res_mpc_fast["th_err"][-1]):.2f}°)')
axs[1, 1].set_title('Fast Motion (T = 4.0 s) — Heading Angle Error', fontsize=11, fontweight='bold')
axs[1, 1].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 1].set_ylabel('Heading Error (°)', fontsize=10)
axs[1, 1].grid(True, linestyle=':', alpha=0.6)
axs[1, 1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.show()

# =======================================================================
# 5. FIGURE 3: ACTUATOR VOLTAGES & CONTACT PATCH WHEEL SLIP
# =======================================================================
fig, axs = plt.subplots(2, 2, figsize=(14, 9))

# Top-Left: Nominal Peak Voltages
axs[0, 0].plot(res_pid_nom['t'], np.max(np.abs(res_pid_nom['V']), axis=1), 'royalblue', lw=2.0, label=f'PID Peak |V| ({np.max(np.abs(res_pid_nom["V"])):.1f} V)')
axs[0, 0].plot(res_mpc_nom['t'], np.max(np.abs(res_mpc_nom['V']), axis=1), 'darkorange', lw=2.0, label=f'MPC Peak |V| ({np.max(np.abs(res_mpc_nom["V"])):.1f} V)')
axs[0, 0].axhline(12.0, color='r', linestyle='--', alpha=0.6, label='Supply Ceiling (±12V)')
axs[0, 0].set_title('Nominal Motion (T = 8.0 s) — Peak Commanded Voltage', fontsize=11, fontweight='bold')
axs[0, 0].set_ylabel('Voltage (V)', fontsize=10)
axs[0, 0].set_ylim(0, 13)
axs[0, 0].grid(True, linestyle=':', alpha=0.6)
axs[0, 0].legend(loc='upper right', fontsize=8.5)

# Top-Right: Fast Peak Voltages
axs[0, 1].plot(res_pid_fast['t'], np.max(np.abs(res_pid_fast['V']), axis=1), 'royalblue', lw=2.0, label=f'PID Peak |V| ({np.max(np.abs(res_pid_fast["V"])):.1f} V)')
axs[0, 1].plot(res_mpc_fast['t'], np.max(np.abs(res_mpc_fast['V']), axis=1), 'darkorange', lw=2.0, label=f'MPC Peak |V| ({np.max(np.abs(res_mpc_fast["V"])):.1f} V)')
axs[0, 1].axhline(12.0, color='r', linestyle='--', alpha=0.6, label='Supply Ceiling (±12V)')
axs[0, 1].set_title('Fast Motion (T = 4.0 s) — Peak Commanded Voltage', fontsize=11, fontweight='bold')
axs[0, 1].set_ylabel('Voltage (V)', fontsize=10)
axs[0, 1].set_ylim(0, 13)
axs[0, 1].grid(True, linestyle=':', alpha=0.6)
axs[0, 1].legend(loc='upper right', fontsize=8.5)

# Bottom-Left: Nominal Peak Slip
axs[1, 0].plot(res_pid_nom['t'], np.max(np.abs(res_pid_nom['vs']), axis=1)*1000.0, 'royalblue', lw=1.8, label=f'PID Peak Slip ({np.max(np.abs(res_pid_nom["vs"]))*1000.0:.1f} mm/s)')
axs[1, 0].plot(res_mpc_nom['t'], np.max(np.abs(res_mpc_nom['vs']), axis=1)*1000.0, 'darkorange', lw=1.8, label=f'MPC Peak Slip ({np.max(np.abs(res_mpc_nom["vs"]))*1000.0:.1f} mm/s)')
axs[1, 0].axhline(5.0, color='r', linestyle='--', alpha=0.7, label='Sliding Limit (5 mm/s)')
axs[1, 0].set_title('Nominal Motion (T = 8.0 s) — Peak Contact Patch Slip Velocity', fontsize=11, fontweight='bold')
axs[1, 0].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 0].set_ylabel('Peak Slip |vs| (mm/s)', fontsize=10)
axs[1, 0].grid(True, linestyle=':', alpha=0.6)
axs[1, 0].legend(loc='upper right', fontsize=8.5)

# Bottom-Right: Fast Peak Slip
axs[1, 1].plot(res_pid_fast['t'], np.max(np.abs(res_pid_fast['vs']), axis=1)*1000.0, 'royalblue', lw=1.8, label=f'PID Peak Slip ({np.max(np.abs(res_pid_fast["vs"]))*1000.0:.1f} mm/s)')
axs[1, 1].plot(res_mpc_fast['t'], np.max(np.abs(res_mpc_fast['vs']), axis=1)*1000.0, 'darkorange', lw=1.8, label=f'MPC Peak Slip ({np.max(np.abs(res_mpc_fast["vs"]))*1000.0:.1f} mm/s)')
axs[1, 1].axhline(5.0, color='r', linestyle='--', alpha=0.7, label='Sliding Limit (5 mm/s)')
axs[1, 1].set_title('Fast Motion (T = 4.0 s) — Peak Contact Patch Slip Velocity', fontsize=11, fontweight='bold')
axs[1, 1].set_xlabel('Simulation Time (s)', fontsize=10)
axs[1, 1].set_ylabel('Peak Slip |vs| (mm/s)', fontsize=10)
axs[1, 1].grid(True, linestyle=':', alpha=0.6)
axs[1, 1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.show()



## ADDED: MPC with the 1 ms friction estimator

Runs after the benchmark above. `'mpc_est'` is `MPCControllerFrictionEstimator`: Dana's MPC with the estimator, same settings; the runner calls the estimator after every 1 ms plant step.

In [ ]:
# ADDED: the MPC with the 1 ms friction estimator on the same two paths, against Dana's runs above
res_est_nom = execute_simulation(sim_params, 'mpc_est', ref_path_slow)
res_est_fast = execute_simulation(sim_params, 'mpc_est', ref_path_fast)

print("=" * 88)
print(f"{'Metric':<32} | {'PID (T=8.0s)':<12} | {'MPC (T=8.0s)':<12} | {'MPC+est 8.0s':<12} | {'PID (T=4.0s)':<12} | {'MPC (T=4.0s)':<12} | {'MPC+est 4.0s':<12}")
print("-" * 120)
runs = (res_pid_nom, res_mpc_nom, res_est_nom, res_pid_fast, res_mpc_fast, res_est_fast)
print(f"{'Maximum Position Error (cm)':<32} | " + " | ".join(f"{np.max(r['dist_err']) * 100:12.2f}" for r in runs))
print(f"{'RMS Position Error (cm)':<32} | " + " | ".join(f"{np.sqrt(np.mean(r['dist_err']**2)) * 100:12.2f}" for r in runs))
print(f"{'Final Position Error (cm)':<32} | " + " | ".join(f"{r['dist_err'][-1] * 100:12.2f}" for r in runs))
print(f"{'Peak Contact Slip (mm/s)':<32} | " + " | ".join(f"{np.max(np.abs(r['vs'])) * 1000:12.1f}" for r in runs))
print(f"{'Execution Time per Step (ms)':<32} | " + " | ".join(f"{r['elapsed'] / len(r['t']) * 1000:12.2f}" for r in runs))
